# Plot light curves

Start with importing necessary packages.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.pylab as pylab
import matplotlib.colors as plc
import astropy.units as u
import astropy.constants as c
from astropy.io import fits
from astropy.time import Time
from sunburn import analysis, tools, hst_observation, spectroscopy, systematics
import os
from scipy.optimize import curve_fit

%config InlineBackend.figure_format = 'retina'

pylab.rcParams['figure.figsize'] = 9.0,6.5
pylab.rcParams['font.size'] = 18
pylab.rcParams['font.family'] = 'Helvetica'

Set datasets and visit objects.

In [ ]:
datasets_visit1 = ['oe8ta1010', 'oe8ta1020', 'oe8ta1030', 'oe8ta1040', 'oe8ta1050']
datasets_visit2 = ['oen502030', 'oen502040', 'oen502050']
datasets_visit3 = ['oen501010', 'oen501020', 'oen501030', 'oen501040', 'oen501050']
datasets_visit4 = ['oen552010', 'oen552020', 'oen552030', 'oen552040', 'oen552050']

visit1 = hst_observation.Visit(datasets_visit1, 'stis', prefix='../data/v1/')  # Modify prefix accordingly, should point to the x1d files directory
visit2 = hst_observation.Visit(datasets_visit2, 'stis', prefix='../data/v2/')  # Modify prefix accordingly, should point to the x1d files directory
visit3 = hst_observation.Visit(datasets_visit3, 'stis', prefix='../data/v3/')  # Modify prefix accordingly, should point to the x1d files directory
visit4 = hst_observation.Visit(datasets_visit4, 'stis', prefix='../data/v4/')  # Modify prefix accordingly, should point to the x1d files directory

Assign split subexposures to each visit.

In [ ]:
visit1.assign_splits('../data/v1/splits10/')
visit2.assign_splits('../data/v2/splits10/')
visit3.assign_splits('../data/v3/splits10/')
visit4.assign_splits('../data/v4/splits10/')

Correct for the instrumental radial-velocity shifts.

In [ ]:
# Correct for RV shifts
rv_shifts = np.loadtxt('rv_shifts.dat')
cs = c.c.to(u.km / u.s).value

v = visit1
rv_corr = - rv_shifts[0:5, 0]
for i in range(len(v.orbit)):
    o = datasets_visit1[i]
    wl_shift = rv_corr[i] / cs * v.orbit[o].wavelength
    v.orbit[o].wavelength += wl_shift

v = visit2
rv_corr = - rv_shifts[5:8, 0]
for i in range(len(v.orbit)):
    o = datasets_visit2[i]
    wl_shift = rv_corr[i] / cs * v.orbit[o].wavelength
    v.orbit[o].wavelength += wl_shift
    
v = visit3
rv_corr = - rv_shifts[8:13, 0]
for i in range(len(v.orbit)):
    o = datasets_visit3[i]
    wl_shift = rv_corr[i] / cs * v.orbit[o].wavelength
    v.orbit[o].wavelength += wl_shift
    
v = visit4
rv_corr = - rv_shifts[13:18, 0]
for i in range(len(v.orbit)):
    o = datasets_visit4[i]
    wl_shift = rv_corr[i] / cs * v.orbit[o].wavelength
    v.orbit[o].wavelength += wl_shift

Interpolate to the same wavelength grid.

In [ ]:
v = visit1
for o in v.orbit:
    main_wl = np.copy(v.orbit[o].wavelength)
    for i in range(len(v.orbit[o].split)):
        v.orbit[o].split[i].wavelength = main_wl

v = visit2
for o in v.orbit:
    main_wl = np.copy(v.orbit[o].wavelength)
    for i in range(len(v.orbit[o].split)):
        v.orbit[o].split[i].wavelength = main_wl
        
v = visit3
for o in v.orbit:
    main_wl = np.copy(v.orbit[o].wavelength)
    for i in range(len(v.orbit[o].split)):
        v.orbit[o].split[i].wavelength = main_wl
        
v = visit4
for o in v.orbit:
    main_wl = np.copy(v.orbit[o].wavelength)
    for i in range(len(v.orbit[o].split)):
        v.orbit[o].split[i].wavelength = main_wl

In [ ]:
# Transit ephemeris, from TFOP
name = 'DS Tuc A b'
period = 8.138224343 * u.d
transit_midpoint = Time(2459040.333471, format='jd')
duration14 = 3.1764 * u.h  # Transit duration betweeen 1st and 4th contacts
duration23 = 0.11818 * u.d  # Transit duration betweeen 2nd and 3rd contacts
rv = 7.8815 * u.km / u.s  # Systemic radial velocity

# The transit object
transit = analysis.Transit(planet_name=name, period=period,
                           transit_midpoint=transit_midpoint, duration14=duration14, duration23=duration23)

line_list = spectroscopy.COSFUVLineList(wavelength_shift=.0, range_factor=1.0).lines

# Systematics de-trending arrays
detrend_factors_v1 = np.loadtxt('10splits_breathing_corr_v1.txt')
detrend_factors_v3 = np.loadtxt('10splits_breathing_corr_v3.txt')
detrend_factors_v4 = np.loadtxt('10splits_breathing_corr_v4.txt')

## Ly-$\alpha$ light curves

Calculate normalized light curves.

In [ ]:
t = 'H I'
x_shift = 0
v_range = np.array([100 - x_shift, 400 - x_shift])

lc1 = analysis.LightCurve(visit1, transit, line_list)
lc1.compute_flux(transition=t, line_index=0, velocity_range=v_range, poisson_regime=True, 
                 detrend_factor_splits=detrend_factors_v1, 
                 recompute_from_splits=True, 
                )
# Normalizing factor for Visit A
norm_1 = np.mean(lc1.integrated_flux)
norm_u_1 = np.sum(lc1.f_unc ** 2) ** 0.5 / 5

lc2 = analysis.LightCurve(visit2, transit, line_list)
lc2.compute_flux(transition=t, line_index=0, velocity_range=v_range, poisson_regime=True, 
                 recompute_from_splits=True, 
                )
norm_2 = (lc2.integrated_flux[0] + lc2.integrated_flux[2]) / 2
norm_u_2 = (lc2.f_unc[0] ** 2 + lc2.f_unc[2] ** 2) ** 0.5 / 2

lc3 = analysis.LightCurve(visit3, transit, line_list)
lc3.compute_flux(transition=t, line_index=0, velocity_range=v_range, poisson_regime=True, 
                 detrend_factor_splits=detrend_factors_v3, 
                 recompute_from_splits=True, 
                )
# Normalizing factor for Visit C
norm_3 = (lc3.integrated_flux[0] + lc3.integrated_flux[4]) / 2
norm_u_3 = (lc3.f_unc[0] ** 2 + lc3.f_unc[4] ** 2) ** 0.5 / 2

lc4 = analysis.LightCurve(visit4, transit, line_list)
lc4.compute_flux(transition=t, line_index=0, velocity_range=v_range, poisson_regime=True, 
                 detrend_factor_splits=detrend_factors_v4, 
                 recompute_from_splits=True, 
                 #doppler_shift_corr=-rv.value,
                 #print_n_pixels=True
                )
# Normalizing factor for Visit D
norm_4 = (lc4.integrated_flux[2] + lc4.integrated_flux[3]) / 2
norm_u_4 = (lc4.f_unc[2] ** 2 + lc4.f_unc[3] ** 2) ** 0.5 / 2

alpha=1
mk_s = 10

lc1.plot(fold=True, plot_splits=False, symbol='v', alpha=0.5, markersize=mk_s,
         log_flux_scale=None, symbol_color='C2', ls='', norm_factor=norm_1,
         norm_uncertainty=norm_u_1, label_choice='Visit A (Jul 2020)'
         )
# lc2.plot(fold=True, plot_splits=True, symbol='o', alpha=alpha, markersize=mk_s,
#          log_flux_scale=-14, symbol_color='C1', ls='', norm_factor=None,#norm_2, 
#          norm_uncertainty=norm_u_2, label_choice='Visit B'
#          )
lc3.plot(fold=True, plot_splits=True, symbol='^', alpha=alpha, markersize=mk_s,
         log_flux_scale=None, symbol_color='C0', ls='', norm_factor=norm_3, 
         norm_uncertainty=norm_u_3, label_choice='Transit 1 (Aug 2022)'
         )
lc4.plot(fold=True, plot_splits=True, symbol='o', alpha=alpha, markersize=mk_s,
         log_flux_scale=None, symbol_color='C1', ls='', norm_factor=norm_4, 
         norm_uncertainty=norm_u_4, label_choice='Transit 2 (Mar 2023)'
         )
plt.ylim(0.65, 1.15)
plt.axhline(y=1.0, ls='--', color='k', alpha=0.5)
plt.title(r'Lyman-$\alpha$ red wing [+%i, +%i] km s$^{-1}$' % (round(v_range[0]), 
                                                                   round(v_range[1])))
plt.legend(fontsize=12, loc=4)
plt.tight_layout()
# plt.savefig('Figure_2b.pdf')
plt.show()

In [ ]:
t = 'H I'
x_shift = 0
v_range = np.array([-150 - x_shift, -50 - x_shift])

lc1 = analysis.LightCurve(visit1, transit, line_list)
lc1.compute_flux(transition=t, line_index=0, velocity_range=v_range, poisson_regime=True, 
                 detrend_factor_splits=detrend_factors_v1, 
                 recompute_from_splits=True, 
                )
# Normalizing factor for Visit A
norm_1 = np.mean(lc1.integrated_flux)
norm_u_1 = np.sum(lc1.f_unc ** 2) ** 0.5 / 5

lc2 = analysis.LightCurve(visit2, transit, line_list)
lc2.compute_flux(transition=t, line_index=0, velocity_range=v_range, poisson_regime=True, 
                 recompute_from_splits=True, 
                )
norm_2 = (lc2.integrated_flux[0] + lc2.integrated_flux[2]) / 2
norm_u_2 = (lc2.f_unc[0] ** 2 + lc2.f_unc[2] ** 2) ** 0.5 / 2

lc3 = analysis.LightCurve(visit3, transit, line_list)
lc3.compute_flux(transition=t, line_index=0, velocity_range=v_range, poisson_regime=True, 
                 detrend_factor_splits=detrend_factors_v3, 
                 recompute_from_splits=True, 
                )
# Normalizing factor for Visit C
norm_3 = (lc3.integrated_flux[0] + lc3.integrated_flux[4]) / 2
norm_u_3 = (lc3.f_unc[0] ** 2 + lc3.f_unc[4] ** 2) ** 0.5 / 2

lc4 = analysis.LightCurve(visit4, transit, line_list)
lc4.compute_flux(transition=t, line_index=0, velocity_range=v_range, poisson_regime=True, 
                 detrend_factor_splits=detrend_factors_v4, 
                 recompute_from_splits=True, 
                 #doppler_shift_corr=-rv.value,
                 #print_n_pixels=True
                )
# Normalizing factor for Visit D
norm_4 = (lc4.integrated_flux[2] + lc4.integrated_flux[3]) / 2
norm_u_4 = (lc4.f_unc[2] ** 2 + lc4.f_unc[3] ** 2) ** 0.5 / 2

alpha=1
mk_s = 10

lc1.plot(fold=True, plot_splits=False, symbol='v', alpha=0.5, markersize=mk_s,
         log_flux_scale=None, symbol_color='C2', ls='', norm_factor=norm_1,
         norm_uncertainty=norm_u_1, label_choice='Visit A (Jul 2020)'
         )
# lc2.plot(fold=True, plot_splits=True, symbol='o', alpha=alpha, markersize=mk_s,
#          log_flux_scale=-14, symbol_color='C1', ls='', norm_factor=None,#norm_2, 
#          norm_uncertainty=norm_u_2, label_choice='Visit B'
#          )
lc3.plot(fold=True, plot_splits=True, symbol='^', alpha=alpha, markersize=mk_s,
         log_flux_scale=None, symbol_color='C0', ls='', norm_factor=norm_3, 
         norm_uncertainty=norm_u_3, label_choice='Transit 1 (Aug 2022)'
         )
lc4.plot(fold=True, plot_splits=True, symbol='o', alpha=alpha, markersize=mk_s,
         log_flux_scale=None, symbol_color='C1', ls='', norm_factor=norm_4, 
         norm_uncertainty=norm_u_4, label_choice='Transit 2 (Mar 2023)'
         )
plt.ylim(0.65, 1.15)
plt.axhline(y=1.0, ls='--', color='k', alpha=0.5)
plt.title(r'Lyman-$\alpha$ inner blue wing [%i, %i] km s$^{-1}$' % (round(v_range[0]), 
                                                                   round(v_range[1])))
plt.legend(fontsize=12, loc=4)
plt.tight_layout()
# plt.savefig('Figure_2b.pdf')
plt.show()

In [ ]:
t = 'H I'
x_shift = 0
v_range = np.array([-500 - x_shift, -150 - x_shift])

lc1 = analysis.LightCurve(visit1, transit, line_list)
lc1.compute_flux(transition=t, line_index=0, velocity_range=v_range, poisson_regime=True, 
                 detrend_factor_splits=detrend_factors_v1, 
                 recompute_from_splits=True, 
                )
# Normalizing factor for Visit A
norm_1 = np.mean(lc1.integrated_flux)
norm_u_1 = np.sum(lc1.f_unc ** 2) ** 0.5 / 5

lc2 = analysis.LightCurve(visit2, transit, line_list)
lc2.compute_flux(transition=t, line_index=0, velocity_range=v_range, poisson_regime=True, 
                 recompute_from_splits=True, 
                )
norm_2 = (lc2.integrated_flux[0] + lc2.integrated_flux[2]) / 2
norm_u_2 = (lc2.f_unc[0] ** 2 + lc2.f_unc[2] ** 2) ** 0.5 / 2

lc3 = analysis.LightCurve(visit3, transit, line_list)
lc3.compute_flux(transition=t, line_index=0, velocity_range=v_range, poisson_regime=True, 
                 detrend_factor_splits=detrend_factors_v3, 
                 recompute_from_splits=True, 
                )
# Normalizing factor for Visit C
norm_3 = (lc3.integrated_flux[0] + lc3.integrated_flux[4]) / 2
norm_u_3 = (lc3.f_unc[0] ** 2 + lc3.f_unc[4] ** 2) ** 0.5 / 2

lc4 = analysis.LightCurve(visit4, transit, line_list)
lc4.compute_flux(transition=t, line_index=0, velocity_range=v_range, poisson_regime=True, 
                 detrend_factor_splits=detrend_factors_v4, 
                 recompute_from_splits=True, 
                 #doppler_shift_corr=-rv.value,
                 #print_n_pixels=True
                )
# Normalizing factor for Visit D
norm_4 = (lc4.integrated_flux[2] + lc4.integrated_flux[3]) / 2
norm_u_4 = (lc4.f_unc[2] ** 2 + lc4.f_unc[3] ** 2) ** 0.5 / 2

alpha=1
mk_s = 10

lc1.plot(fold=True, plot_splits=False, symbol='v', alpha=0.5, markersize=mk_s,
         log_flux_scale=None, symbol_color='C2', ls='', norm_factor=norm_1,
         norm_uncertainty=norm_u_1, label_choice='Visit A (Jul 2020)'
         )
# lc2.plot(fold=True, plot_splits=True, symbol='o', alpha=alpha, markersize=mk_s,
#          log_flux_scale=-14, symbol_color='C1', ls='', norm_factor=None,#norm_2, 
#          norm_uncertainty=norm_u_2, label_choice='Visit B'
#          )
lc3.plot(fold=True, plot_splits=True, symbol='^', alpha=alpha, markersize=mk_s,
         log_flux_scale=None, symbol_color='C0', ls='', norm_factor=norm_3, 
         norm_uncertainty=norm_u_3, label_choice='Transit 1 (Aug 2022)'
         )
lc4.plot(fold=True, plot_splits=True, symbol='o', alpha=alpha, markersize=mk_s,
         log_flux_scale=None, symbol_color='C1', ls='', norm_factor=norm_4, 
         norm_uncertainty=norm_u_4, label_choice='Transit 2 (Mar 2023)'
         )
plt.ylim(0.65, 1.15)
plt.axhline(y=1.0, ls='--', color='k', alpha=0.5)
plt.title(r'Lyman-$\alpha$ outer blue wing [%i, %i] km s$^{-1}$' % (round(v_range[0]), 
                                                                   round(v_range[1])))
plt.legend(fontsize=12, loc=4)
plt.tight_layout()
# plt.savefig('Figure_2b.pdf')
plt.show()

## Metals light curves

In [ ]:
t = 'Si III'
x_shift = 0
v_range = np.array([0 - x_shift, 100 - x_shift])

lc1 = analysis.LightCurve(visit1, transit, line_list)
lc1.compute_flux(transition=t, line_index=0, velocity_range=v_range, poisson_regime=True, 
                 detrend_factor_splits=detrend_factors_v1, 
                 recompute_from_splits=True, 
                )

norm_1 = np.mean(lc1.integrated_flux)
norm_u_1 = np.sum(lc1.f_unc ** 2) ** 0.5 / 5

lc2 = analysis.LightCurve(visit2, transit, line_list)
lc2.compute_flux(transition=t, line_index=0, velocity_range=v_range, poisson_regime=True, 
                 recompute_from_splits=True, 
                )
norm_2 = (lc2.integrated_flux[0] + lc2.integrated_flux[2]) / 2
norm_u_2 = (lc2.f_unc[0] ** 2 + lc2.f_unc[2] ** 2) ** 0.5 / 2

lc3 = analysis.LightCurve(visit3, transit, line_list)
lc3.compute_flux(transition=t, line_index=0, velocity_range=v_range, poisson_regime=True, 
                 detrend_factor_splits=detrend_factors_v3, 
                 recompute_from_splits=True, 
                )

norm_3 = (lc3.integrated_flux[0] + lc3.integrated_flux[4]) / 2
norm_u_3 = (lc3.f_unc[0] ** 2 + lc3.f_unc[4] ** 2) ** 0.5 / 2

lc4 = analysis.LightCurve(visit4, transit, line_list)
lc4.compute_flux(transition=t, line_index=0, velocity_range=v_range, poisson_regime=True, 
                 detrend_factor_splits=detrend_factors_v4, 
                 recompute_from_splits=True, 
                )
norm_4 = (lc4.integrated_flux[2] + lc4.integrated_flux[3]) / 2
norm_u_4 = (lc4.f_unc[2] ** 2 + lc4.f_unc[3] ** 2) ** 0.5 / 2

alpha=1
mk_s = 10

lc1.plot(fold=True, plot_splits=False, symbol='v', alpha=0.5, markersize=mk_s,
         log_flux_scale=None, symbol_color='C2', ls='', norm_factor=norm_1,
         norm_uncertainty=norm_u_1, label_choice='Visit A (Jul 2020)'
         )
# lc2.plot(fold=True, plot_splits=True, symbol='o', alpha=alpha, markersize=mk_s,
#          log_flux_scale=-14, symbol_color='C1', ls='', norm_factor=None,#norm_2, 
#          norm_uncertainty=norm_u_2, label_choice='Visit B'
#          )
lc3.plot(fold=True, plot_splits=True, symbol='^', alpha=alpha, markersize=mk_s,
         log_flux_scale=None, symbol_color='C0', ls='', norm_factor=norm_3, 
         norm_uncertainty=norm_u_3, label_choice='Visit C (Aug 2022)'
         )
lc4.plot(fold=True, plot_splits=True, symbol='o', alpha=alpha, markersize=mk_s,
         log_flux_scale=None, symbol_color='C1', ls='', norm_factor=norm_4, 
         norm_uncertainty=norm_u_4, label_choice='Visit D (Mar 2023)'
         )
# plt.ylim(0.65, 1.35)
plt.ylim(0.65, 1.15)
plt.axhline(y=1.0, ls='--', color='k', alpha=0.5)
plt.title(r'Si III red wing [+%i, +%i] km s$^{-1}$' % (round(v_range[0]), 
                                                                   round(v_range[1])))
# plt.title(r'Lyman-$\alpha$ red wing transit light curve')
plt.legend(fontsize=12, loc=4)
plt.tight_layout()
# plt.savefig('SiIII_red_lc_norm.pdf')
plt.show()

In [ ]:
t = 'Si III'
x_shift = 0
v_range = np.array([-100 - x_shift, 0 - x_shift])

lc1 = analysis.LightCurve(visit1, transit, line_list)
lc1.compute_flux(transition=t, line_index=0, velocity_range=v_range, poisson_regime=True, 
                 detrend_factor_splits=detrend_factors_v1, 
                 recompute_from_splits=True, 
                )
norm_1 = np.mean(lc1.integrated_flux)
norm_u_1 = np.sum(lc1.f_unc ** 2) ** 0.5 / 5

lc2 = analysis.LightCurve(visit2, transit, line_list)
lc2.compute_flux(transition=t, line_index=0, velocity_range=v_range, poisson_regime=True, 
                 recompute_from_splits=True, 
                )
norm_2 = (lc2.integrated_flux[0] + lc2.integrated_flux[2]) / 2
norm_u_2 = (lc2.f_unc[0] ** 2 + lc2.f_unc[2] ** 2) ** 0.5 / 2

lc3 = analysis.LightCurve(visit3, transit, line_list)
lc3.compute_flux(transition=t, line_index=0, velocity_range=v_range, poisson_regime=True, 
                 detrend_factor_splits=detrend_factors_v3, 
                 recompute_from_splits=True, 
                )

norm_3 = (lc3.integrated_flux[0] + lc3.integrated_flux[4]) / 2
norm_u_3 = (lc3.f_unc[0] ** 2 + lc3.f_unc[4] ** 2) ** 0.5 / 2

lc4 = analysis.LightCurve(visit4, transit, line_list)
lc4.compute_flux(transition=t, line_index=0, velocity_range=v_range, poisson_regime=True, 
                 detrend_factor_splits=detrend_factors_v4, 
                 recompute_from_splits=True, 
                 #doppler_shift_corr=-rv.value,
                 #print_n_pixels=True
                )
norm_4 = (lc4.integrated_flux[2] + lc4.integrated_flux[3]) / 2
norm_u_4 = (lc4.f_unc[2] ** 2 + lc4.f_unc[3] ** 2) ** 0.5 / 2

alpha=1
mk_s = 10

lc1.plot(fold=True, plot_splits=False, symbol='v', alpha=0.5, markersize=mk_s,
         log_flux_scale=None, symbol_color='C2', ls='', norm_factor=norm_1,
         norm_uncertainty=norm_u_1, label_choice='Visit A (Jul 2020)'
         )
# lc2.plot(fold=True, plot_splits=True, symbol='o', alpha=alpha, markersize=mk_s,
#          log_flux_scale=-14, symbol_color='C1', ls='', norm_factor=None,#norm_2, 
#          norm_uncertainty=norm_u_2, label_choice='Visit B'
#          )
lc3.plot(fold=True, plot_splits=True, symbol='^', alpha=alpha, markersize=mk_s,
         log_flux_scale=None, symbol_color='C0', ls='', norm_factor=norm_3, 
         norm_uncertainty=norm_u_3, label_choice='Visit C (Aug 2022)'
         )
lc4.plot(fold=True, plot_splits=True, symbol='o', alpha=alpha, markersize=mk_s,
         log_flux_scale=None, symbol_color='C1', ls='', norm_factor=norm_4, 
         norm_uncertainty=norm_u_4, label_choice='Visit D (Mar 2023)'
         )
plt.ylim(0.65, 1.15)
plt.axhline(y=1.0, ls='--', color='k', alpha=0.5)
plt.title(r'Si III blue wing [%i, %i] km s$^{-1}$' % (round(v_range[0]), 
                                                                   round(v_range[1])))
plt.tight_layout()
# plt.savefig('SiIII_blue_lc_norm.pdf')
plt.show()

In [ ]:
t = 'Si III'
x_shift = 0
v_range = np.array([-100 - x_shift, 100 - x_shift])

lc1 = analysis.LightCurve(visit1, transit, line_list)
lc1.compute_flux(transition=t, line_index=0, velocity_range=v_range, poisson_regime=True, 
                 detrend_factor_splits=detrend_factors_v1, 
                 recompute_from_splits=True, 
                )
norm_1 = np.mean(lc1.integrated_flux)
norm_u_1 = np.sum(lc1.f_unc ** 2) ** 0.5 / 5

lc2 = analysis.LightCurve(visit2, transit, line_list)
lc2.compute_flux(transition=t, line_index=0, velocity_range=v_range, poisson_regime=True, 
                 recompute_from_splits=True, 
                )
norm_2 = (lc2.integrated_flux[0] + lc2.integrated_flux[2]) / 2
norm_u_2 = (lc2.f_unc[0] ** 2 + lc2.f_unc[2] ** 2) ** 0.5 / 2

lc3 = analysis.LightCurve(visit3, transit, line_list)
lc3.compute_flux(transition=t, line_index=0, velocity_range=v_range, poisson_regime=True, 
                 detrend_factor_splits=detrend_factors_v3, 
                 recompute_from_splits=True, 
                )
norm_3 = (lc3.integrated_flux[0] + lc3.integrated_flux[4]) / 2
norm_u_3 = (lc3.f_unc[0] ** 2 + lc3.f_unc[4] ** 2) ** 0.5 / 2

lc4 = analysis.LightCurve(visit4, transit, line_list)
lc4.compute_flux(transition=t, line_index=0, velocity_range=v_range, poisson_regime=True, 
                 detrend_factor_splits=detrend_factors_v4, 
                 recompute_from_splits=True, 
                )
norm_4 = (lc4.integrated_flux[2] + lc4.integrated_flux[3]) / 2
norm_u_4 = (lc4.f_unc[2] ** 2 + lc4.f_unc[3] ** 2) ** 0.5 / 2

alpha=1
mk_s = 10

lc1.plot(fold=True, plot_splits=False, symbol='v', alpha=0.5, markersize=mk_s,
         log_flux_scale=None, symbol_color='C2', ls='', norm_factor=norm_1,
         norm_uncertainty=norm_u_1, label_choice='Visit A (Jul 2020)'
         )
# lc2.plot(fold=True, plot_splits=True, symbol='o', alpha=alpha, markersize=mk_s,
#          log_flux_scale=-14, symbol_color='C1', ls='', norm_factor=None,#norm_2, 
#          norm_uncertainty=norm_u_2, label_choice='Visit B'
#          )
lc3.plot(fold=True, plot_splits=True, symbol='^', alpha=alpha, markersize=mk_s,
         log_flux_scale=None, symbol_color='C0', ls='', norm_factor=norm_3, 
         norm_uncertainty=norm_u_3, label_choice='Visit C (Aug 2022)'
         )
lc4.plot(fold=True, plot_splits=True, symbol='o', alpha=alpha, markersize=mk_s,
         log_flux_scale=None, symbol_color='C1', ls='', norm_factor=norm_4, 
         norm_uncertainty=norm_u_4, label_choice='Visit D (Mar 2023)'
         )
plt.ylim(0.65, 1.15)
plt.axhline(y=1.0, ls='--', color='k', alpha=0.5)
plt.title(r'Si III full line [%i, %i] km s$^{-1}$' % (round(v_range[0]), 
                                                                   round(v_range[1])))
plt.tight_layout()
# plt.savefig('SiIII_full_lc_norm.pdf')
plt.show()

In [ ]:
t = 'N V'
x_shift = 0
l_ind = 0
v_range = np.array([-100 - x_shift, 100 - x_shift])

lc1 = analysis.LightCurve(visit1, transit, line_list)
lc1.compute_flux(transition=t, line_index=l_ind, velocity_range=v_range, poisson_regime=True, 
                 detrend_factor_splits=detrend_factors_v1, 
                 recompute_from_splits=True, 
                )
norm_1 = np.mean(lc1.integrated_flux)
norm_u_1 = np.sum(lc1.f_unc ** 2) ** 0.5 / 5

lc2 = analysis.LightCurve(visit2, transit, line_list)
lc2.compute_flux(transition=t, line_index=l_ind, velocity_range=v_range, poisson_regime=True, 
                 recompute_from_splits=True, 
                )
norm_2 = (lc2.integrated_flux[0] + lc2.integrated_flux[2]) / 2
norm_u_2 = (lc2.f_unc[0] ** 2 + lc2.f_unc[2] ** 2) ** 0.5 / 2

lc3 = analysis.LightCurve(visit3, transit, line_list)
lc3.compute_flux(transition=t, line_index=l_ind, velocity_range=v_range, poisson_regime=True, 
                 detrend_factor_splits=detrend_factors_v3, 
                 recompute_from_splits=True, 
                )
norm_3 = (lc3.integrated_flux[0] + lc3.integrated_flux[4]) / 2
norm_u_3 = (lc3.f_unc[0] ** 2 + lc3.f_unc[4] ** 2) ** 0.5 / 2

lc4 = analysis.LightCurve(visit4, transit, line_list)
lc4.compute_flux(transition=t, line_index=l_ind, velocity_range=v_range, poisson_regime=True, 
                 detrend_factor_splits=detrend_factors_v4, 
                 recompute_from_splits=True, 
                )
norm_4 = (lc4.integrated_flux[2] + lc4.integrated_flux[3]) / 2
norm_u_4 = (lc4.f_unc[2] ** 2 + lc4.f_unc[3] ** 2) ** 0.5 / 2

alpha=1
mk_s = 10

lc1.plot(fold=True, plot_splits=False, symbol='v', alpha=0.5, markersize=mk_s,
         log_flux_scale=None, symbol_color='C2', ls='', norm_factor=norm_1,
         norm_uncertainty=norm_u_1, label_choice='Visit A (Jul 2020)'
         )
# lc2.plot(fold=True, plot_splits=True, symbol='o', alpha=alpha, markersize=mk_s,
#          log_flux_scale=-14, symbol_color='C1', ls='', norm_factor=None,#norm_2, 
#          norm_uncertainty=norm_u_2, label_choice='Visit B'
#          )
lc3.plot(fold=True, plot_splits=True, symbol='^', alpha=alpha, markersize=mk_s,
         log_flux_scale=None, symbol_color='C0', ls='', norm_factor=norm_3, 
         norm_uncertainty=norm_u_3, label_choice='Visit C (Aug 2022)'
         )
lc4.plot(fold=True, plot_splits=True, symbol='o', alpha=alpha, markersize=mk_s,
         log_flux_scale=None, symbol_color='C1', ls='', norm_factor=norm_4, 
         norm_uncertainty=norm_u_4, label_choice='Visit D (Mar 2023)'
         )
plt.ylim(0.65, 1.15)
plt.axhline(y=1.0, ls='--', color='k', alpha=0.5)
plt.title(r'N V @1239 ${\rm \AA}$ full line [%i, %i] km s$^{-1}$' % (round(v_range[0]), 
                                                                   round(v_range[1])))
plt.tight_layout()
# plt.savefig('NV_0_full_lc_norm.pdf')
plt.show()

In [ ]:
t = 'N V'
x_shift = 0
l_ind = 1
v_range = np.array([-100 - x_shift, 100 - x_shift])

lc1 = analysis.LightCurve(visit1, transit, line_list)
lc1.compute_flux(transition=t, line_index=l_ind, velocity_range=v_range, poisson_regime=True, 
                 detrend_factor_splits=detrend_factors_v1, 
                 recompute_from_splits=True, 
                )
norm_1 = np.mean(lc1.integrated_flux)
norm_u_1 = np.sum(lc1.f_unc ** 2) ** 0.5 / 5

lc2 = analysis.LightCurve(visit2, transit, line_list)
lc2.compute_flux(transition=t, line_index=l_ind, velocity_range=v_range, poisson_regime=True, 
                 recompute_from_splits=True, 
                )
norm_2 = (lc2.integrated_flux[0] + lc2.integrated_flux[2]) / 2
norm_u_2 = (lc2.f_unc[0] ** 2 + lc2.f_unc[2] ** 2) ** 0.5 / 2

lc3 = analysis.LightCurve(visit3, transit, line_list)
lc3.compute_flux(transition=t, line_index=l_ind, velocity_range=v_range, poisson_regime=True, 
                 detrend_factor_splits=detrend_factors_v3, 
                 recompute_from_splits=True, 
                )
norm_3 = (lc3.integrated_flux[0] + lc3.integrated_flux[4]) / 2
norm_u_3 = (lc3.f_unc[0] ** 2 + lc3.f_unc[4] ** 2) ** 0.5 / 2

lc4 = analysis.LightCurve(visit4, transit, line_list)
lc4.compute_flux(transition=t, line_index=l_ind, velocity_range=v_range, poisson_regime=True, 
                 detrend_factor_splits=detrend_factors_v4, 
                 recompute_from_splits=True, 
                )
norm_4 = (lc4.integrated_flux[2] + lc4.integrated_flux[3]) / 2
norm_u_4 = (lc4.f_unc[2] ** 2 + lc4.f_unc[3] ** 2) ** 0.5 / 2

alpha=1
mk_s = 10

lc1.plot(fold=True, plot_splits=False, symbol='v', alpha=0.5, markersize=mk_s,
         log_flux_scale=None, symbol_color='C2', ls='', norm_factor=norm_1,
         norm_uncertainty=norm_u_1, label_choice='Visit A (Jul 2020)'
         )
# lc2.plot(fold=True, plot_splits=True, symbol='o', alpha=alpha, markersize=mk_s,
#          log_flux_scale=-14, symbol_color='C1', ls='', norm_factor=None,#norm_2, 
#          norm_uncertainty=norm_u_2, label_choice='Visit B'
#          )
lc3.plot(fold=True, plot_splits=True, symbol='^', alpha=alpha, markersize=mk_s,
         log_flux_scale=None, symbol_color='C0', ls='', norm_factor=norm_3, 
         norm_uncertainty=norm_u_3, label_choice='Visit C (Aug 2022)'
         )
lc4.plot(fold=True, plot_splits=True, symbol='o', alpha=alpha, markersize=mk_s,
         log_flux_scale=None, symbol_color='C1', ls='', norm_factor=norm_4, 
         norm_uncertainty=norm_u_4, label_choice='Visit D (Mar 2023)'
         )
plt.ylim(0.65, 1.15)
plt.axhline(y=1.0, ls='--', color='k', alpha=0.5)
plt.title(r'N V @1243 ${\rm \AA}$ full line [%i, %i] km s$^{-1}$' % (round(v_range[0]), 
                                                                   round(v_range[1])))
plt.tight_layout()
# plt.savefig('NV_1_full_lc_norm.pdf')
plt.show()